# Forecast Ongoing/Open Prospects


In [1]:
import pandas as pd
import joblib

df_open_encoded = pd.read_csv('../../data/processed/open_prospects_dataset.csv')
best_model = joblib.load('../../models/xgb_churn_model.pkl')
X_train, _, _, _, _, _ = joblib.load('../../models/train_test_data.pkl')
print(f'Loaded model and Open Prospects data: {df_open_encoded.shape}')


Loaded model and Open Prospects data: (37604, 199)


# 7. Predicting on Open Prospects
Processing the `open_prospects` dataset through the identical ML pipeline and predicting the probability of churn for current ongoing engagements.

In [2]:
# 7.3 Feature Alignment
final_features = list(X_train.columns)  # Extracted from Section 4

# Reindex strictly aligns the open_prospects to the exact columns the model was trained on
# Missing dummy variables will be filled with 0
X_open = df_open_encoded.reindex(columns=final_features, fill_value=0)
X_open = X_open.astype(float)

print(f'Final engineered X_open shape: {X_open.shape}')

Final engineered X_open shape: (37604, 120)


In [3]:
# 7.4 Execute Churn Predictions
# Fetch our optimized XGBoost Model object
best_model = best_model

# Predict
open_preds = best_model.predict(X_open)
open_probs = best_model.predict_proba(X_open)[:, 1] # Prob of Class 1 (Churned)

open_results = pd.DataFrame({
    'co_ref': df_open_encoded['co_ref'],
    'predicted_class': open_preds,  # 1 = Churn, 0 = Retained
    'churn_probability': open_probs * 100
})

# Map classes
open_results['forecast'] = open_results['predicted_class'].map({1: 'High Risk (Churn)', 0: 'Safe (Retained)'})

print('=================================')
print('       FORECAST SUMMARY')
print('=================================')
print(f"Total Engagements: {len(open_results):,}")
print(f"Flagged for Churn: {int(open_results['predicted_class'].sum()):,} ({(open_results['predicted_class'].sum()/len(open_results))*100:.1f}%)")
print('=================================')

       FORECAST SUMMARY
Total Engagements: 37,604
Flagged for Churn: 8,402 (22.3%)


In [4]:
# 7.5 Display High-Risk Prospects (Intervention List)
high_risk = open_results.sort_values(by='churn_probability', ascending=False)

print('\n--- TOP 20 HIGHEST RISK PROSPECTS (Intervention Recommended) ---')
display(high_risk.head(20))

high_risk.to_csv('../../data/processed/open_prospects_churn_predictions.csv')


--- TOP 20 HIGHEST RISK PROSPECTS (Intervention Recommended) ---


,co_ref,predicted_class,churn_probability,forecast
37321,RC4648,1,99.842201,High Risk (Churn)
36578,SQ8515,1,99.832840,High Risk (Churn)
3204,KU5123,1,99.826088,High Risk (Churn)
3165,AH3231,1,99.790062,High Risk (Churn)
37322,RC4648,1,99.780182,High Risk (Churn)
37326,RC4648,1,99.779594,High Risk (Churn)
37323,RC4648,1,99.778793,High Risk (Churn)
37324,RC4648,1,99.778793,High Risk (Churn)
3207,KU5123,1,99.768387,High Risk (Churn)
3163,AH3231,1,99.766708,High Risk (Churn)
